In [1]:
!pip install pandas requests tqdm

In [2]:
import pandas as pd
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# NEWSDATA_KEY = "pub_7ed98964b18745d3b7d91e3e7014559f"

# MARKETAUX_KEY = "uGBA5GdIKVR2QSCj3R8HFeTSX6unCRfQC9NYd2g1"

# ALPHA_KEY = "R2J9I9NZC4TK0SRK"
# session = requests.Session()

In [4]:
# =====================================
# LOAD SIGNAL DATASET
# =====================================

signals = pd.read_csv(
"/content/drive/MyDrive/AI_MANUAL_PROJECT/datasets/manual_signal_dataset.csv"
)

signals["time"] = pd.to_datetime(
    signals["time"]
)

signals = signals[
    signals["time"] >= "2022-01-01"
].reset_index(drop=True)

print(signals.shape)

(80232, 31)


In [5]:
# =====================================
# LOAD CRYPTO NEWS DATASET
# =====================================

news = pd.read_csv(
"/content/drive/MyDrive/AI_MANUAL_PROJECT/datasets/cryptonews.csv"
)

news["date"] = pd.to_datetime(

    news["date"],

    errors="coerce"

)

print(news.shape)

print(news.columns)

(31037, 7)
Index(['date', 'sentiment', 'source', 'subject', 'text', 'title', 'url'], dtype='object')


In [6]:
news["news_text"] = (

    news["title"].fillna("")

    +

    " "

    +

    news["text"].fillna("")

)

In [7]:
btc_mask = news["news_text"].str.contains(
    "bitcoin|btc",
    case=False,
    na=False
)

eth_mask = news["news_text"].str.contains(
    "ethereum|eth",
    case=False,
    na=False
)

sol_mask = news["news_text"].str.contains(
    "solana|sol",
    case=False,
    na=False
)

In [8]:
news["pair"] = "GENERAL"

news.loc[
    btc_mask,
    "pair"
] = "BTC-USDT"

news.loc[
    eth_mask,
    "pair"
] = "ETH-USDT"

news.loc[
    sol_mask,
    "pair"
] = "SOL-USDT"

In [9]:
news = news[
[
    "date",
    "pair",
    "title",
    "text",
    "sentiment",
    "source"
]
]

print(news.shape)

news.head()

(31037, 6)


,date,pair,title,text,sentiment,source
0,2023-12-19 06:40:41,BTC-USDT,Grayscale CEO Calls for Simultaneous Approval ...,Grayscale CEO Michael Sonnenshein believes the...,"{'class': 'negative', 'polarity': -0.1, 'subje...",CryptoNews
1,2023-12-19 06:03:24,GENERAL,Indian Government is Actively Collaborating Wi...,"In an exclusive interview with CryptoNews, Man...","{'class': 'neutral', 'polarity': 0.0, 'subject...",CryptoNews
2,2023-12-19 05:55:14,GENERAL,Judge Approves Settlement: Binance to Pay $1.5...,According to the Federal Court ruling on Decem...,"{'class': 'positive', 'polarity': 0.05, 'subje...",CryptoNews
3,2023-12-19 05:35:26,GENERAL,Why a gold rush for inscriptions has broken ha...,Some suggest EVM inscriptions are the latest w...,"{'class': 'positive', 'polarity': 0.5, 'subjec...",CoinTelegraph
4,2023-12-19 05:31:08,ETH-USDT,‘Concerning precedent’ — bloXroute Labs' MEV r...,A decision by bloXroute Labs to start censorin...,"{'class': 'neutral', 'polarity': 0.0, 'subject...",CoinTelegraph


In [10]:
news_dataset = []

In [11]:
for _, row in tqdm(
        signals.iterrows(),
        total=len(signals)
):

    signal_time = row["time"]

    pair = row["pair"]

    temp_news = news[

        (news["pair"] == pair)

        &

        (

            abs(
                news["date"]
                -
                signal_time
            )

            <=

            pd.Timedelta(
                hours=24
            )

        )

    ]

    temp_news = temp_news.sort_values(
        "date",
        ascending=False
    )

    articles = temp_news.head(3)

    headlines = articles["title"].tolist()

    texts = articles["text"].tolist()

    sentiments = articles["sentiment"].tolist()

    while len(headlines) < 3:

        headlines.append(
            "NO_NEWS"
        )

        texts.append(
            "NO_NEWS"
        )

        sentiments.append(
            "neutral"
        )

    new_row = row.to_dict()

    new_row["headline_1"] = headlines[0]
    new_row["summary_1"] = texts[0]
    new_row["sentiment_1"] = sentiments[0]

    new_row["headline_2"] = headlines[1]
    new_row["summary_2"] = texts[1]
    new_row["sentiment_2"] = sentiments[1]

    new_row["headline_3"] = headlines[2]
    new_row["summary_3"] = texts[2]
    new_row["sentiment_3"] = sentiments[2]

    new_row["article_count"] = len(
        articles
    )

    new_row["news_found"] = (
        len(articles) > 0
    )

    news_dataset.append(
        new_row
    )

100%|██████████| 80232/80232 [08:07<00:00, 164.42it/s]


In [12]:
manual_news_df = pd.DataFrame(
    news_dataset
)

print(
    manual_news_df.shape
)

(80232, 42)


In [13]:
manual_news_df.to_csv(

"/content/drive/MyDrive/AI_MANUAL_PROJECT/datasets/manual_news_dataset.csv",

index=False

)

print(
    "manual_news_dataset.csv saved"
)

manual_news_dataset.csv saved


In [14]:
print(
manual_news_df["news_found"]
.value_counts()
)

print()

print(
manual_news_df["article_count"]
.value_counts()
)

news_found
False    47072
True     33160
Name: count, dtype: int64

article_count
0    47072
3    30779
2     1457
1      924
Name: count, dtype: int64
